# 🔐 nanochat Mini Demo — ROT13 in ~1 Minute

This notebook trains a **tiny 2-layer GPT** (~123 K params) to learn the ROT13 cipher from scratch.

**ROT13** shifts each letter 13 positions in the alphabet:
- `hello` → `uryyb`
- `world` → `jbeyq`
- It is its own inverse: `rot13(rot13(x)) == x`

Uses a **character-level (ASCII) tokenizer** — zero dataset download needed.

**Expected training time: ~70 s on Apple Silicon MPS · ~50 s on CUDA · ~4 min on CPU**

In [ ]:
import sys, math, random, time
import torch

sys.path.insert(0, '../..')   # make nanochat importable when running from dev/rot13/
from nanochat.gpt import GPT, GPTConfig

if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'Device: {device}')

## 1. Build the tiny model

| Param | Value | Why |
|---|---|---|
| `vocab_size` | 128 | Full ASCII — no tokenizer training needed |
| `n_layer` | 2 | Minimal depth |
| `n_head` | 2 | 2 attention heads |
| `n_embd` | 64 | 64-dim residual stream |
| `sequence_len` | 32 | Longest example is ~20 chars |
| `window_pattern` | `"L"` | Full causal attention at both layers |

In [ ]:
config = GPTConfig(
    vocab_size=128,
    n_layer=2,
    n_head=2,
    n_kv_head=2,
    n_embd=64,
    sequence_len=32,
    window_pattern='L',
)

model = GPT(config).to(device)
model.init_weights()

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters : {n_params:,}')
print(f'FLOPs/token: {model.estimate_flops():,}')
print()
for k, v in model.num_scaling_params().items():
    print(f'  {k:<22}: {v:,}')

## 2. The task: ROT13 cipher

Training sequences look like:
```
rot13:hello=uryyb\n
```
At inference time we feed the prompt `rot13:hello=` and let the model complete it.

In [ ]:
def rot13_char(c):
    if 'a' <= c <= 'z':
        return chr((ord(c) - ord('a') + 13) % 26 + ord('a'))
    if 'A' <= c <= 'Z':
        return chr((ord(c) - ord('A') + 13) % 26 + ord('A'))
    return c

def rot13(s):       return ''.join(rot13_char(c) for c in s)
def encode(s):      return [ord(c) for c in s]
def decode(tokens): return ''.join(chr(t) for t in tokens if 0 <= t < 128)

assert rot13('hello') == 'uryyb'
assert rot13(rot13('hello')) == 'hello'   # self-inverse
print('ROT13 sanity checks passed ✓')
for w in ['hello', 'world', 'nanochat', 'transformer']:
    print(f'  {w:14} → {rot13(w)}')

ALPHABET = 'abcdefghijklmnopqrstuvwxyz'

def random_word(min_len=3, max_len=8):
    return ''.join(random.choice(ALPHABET) for _ in range(random.randint(min_len, max_len)))

def make_example(word):
    return f'rot13:{word}={rot13(word)}\n'

random.seed(42)
print()
print('Example training sequences:')
for _ in range(5):
    ex = make_example(random_word())
    print(f'  {ex!r:<38}  ({len(ex)} tokens)')

## 3. Training

- **Loss**: cross-entropy on the output suffix only (tokens after `=`)
- **Optimizer**: AdamW with cosine LR schedule (nanochat uses Muon in production)
- **Steps**: 3 000 · Batch: 128 examples/step

In [ ]:
def build_batch(B, T, device):
    """
    Build a batch of ROT13 training examples.
    inputs  shape (B, T): token ids (padded with 0)
    targets shape (B, T): next-token targets; -1 = ignore (prompt region)
    """
    inputs  = torch.zeros(B, T, dtype=torch.long)
    targets = torch.full((B, T), -1, dtype=torch.long)
    for i in range(B):
        word  = random_word()
        seq   = make_example(word)          # 'rot13:abc=nop\n'
        toks  = encode(seq)
        L     = min(len(toks), T)
        inputs[i, :L] = torch.tensor(toks[:L])
        # At position of '=', the model should start predicting output chars.
        # plen = len('rot13:') + len(word) + len('=') = 7 + len(word)
        plen  = len(f'rot13:{word}=')
        for t in range(plen - 1, L - 1):
            targets[i, t] = toks[t + 1]
    return inputs.to(device), targets.to(device)

In [ ]:
NUM_STEPS  = 3000
BATCH_SIZE = 128
SEQ_LEN    = 32
BASE_LR    = 5e-3
WARMUP     = 100    # linear ramp from 1e-4 to BASE_LR

optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=0.01)

def get_lr(step):
    if step < WARMUP:
        return 1e-4 + (BASE_LR - 1e-4) * step / WARMUP
    progress = (step - WARMUP) / max(1, NUM_STEPS - WARMUP)
    return BASE_LR * max(0.03, 0.5 * (1.0 + math.cos(math.pi * progress)))

model.train()
torch.manual_seed(0)
random.seed(0)
losses = []
t0 = time.time()

for step in range(NUM_STEPS):
    lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    inp, tgt = build_batch(BATCH_SIZE, SEQ_LEN, device)
    loss = model(inp, targets=tgt)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    losses.append(loss.item())

    if step % 500 == 0 or step == NUM_STEPS - 1:
        recent = sum(losses[-50:]) / len(losses[-50:])
        print(f'step {step:4d}/{NUM_STEPS}  loss={recent:.4f}  lr={lr:.5f}  elapsed={time.time()-t0:.0f}s')

print(f'\nTotal: {time.time()-t0:.0f}s')

In [ ]:
try:
    import matplotlib.pyplot as plt
    w = 30
    sm = [sum(losses[max(0,i-w):i+1]) / len(losses[max(0,i-w):i+1]) for i in range(len(losses))]
    plt.figure(figsize=(10, 3))
    plt.plot(losses, alpha=0.25, label='raw')
    plt.plot(sm, label='smoothed (30-step)')
    plt.xlabel('step'); plt.ylabel('cross-entropy loss')
    plt.title('ROT13 — nanochat mini (123 K params)')
    plt.legend(); plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not available — skipping plot')

## 4. Evaluation

In [ ]:
model.eval()

def predict_rot13(word, temperature=0.0):
    prompt = encode(f'rot13:{word}=')
    out = []
    for tok in model.generate(prompt, max_tokens=len(word) + 2, temperature=temperature):
        if tok == ord('\n'):
            break
        out.append(tok)
        if len(out) > len(word):
            break
    return decode(out)

# 200 held-out random words
random.seed(999)
test_words = [random_word() for _ in range(200)]
correct = sum(predict_rot13(w) == rot13(w) for w in test_words)
print(f'Accuracy: {correct}/{len(test_words)} = {correct/len(test_words)*100:.1f}%')

print()
print(f'{"word":<12} {"predicted":<12} {"expected":<12}')
print('-' * 42)
random.seed(42)
for word in [random_word() for _ in range(15)]:
    pred = predict_rot13(word)
    ok   = '✓' if pred == rot13(word) else '✗'
    print(f'{word:<12} {pred:<12} {rot13(word):<12} {ok}')

## 5. Try it yourself

In [ ]:
def demo(word):
    pred = predict_rot13(word.lower())
    exp  = rot13(word.lower())
    ok   = '✓' if pred == exp else '✗'
    print(f'rot13({word!r:14}) = {pred!r:14}  [expected {exp!r}] {ok}')

demo('hello')
demo('world')
demo('nanochat')
demo('transformer')
demo('openai')
demo('python')